In [22]:
import pandas as pd
from nltk.corpus import stopwords
import string
from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from gensim.models import Word2Vec
import numpy as np

In [2]:
stop_words = set(stopwords.words('english'))

In [3]:
#Nettoyage des données
def text_process(mess):
    lower_mess = mess.lower()
    nopunc = [char for char in lower_mess if char not in string.punctuation]
    nopunc = ''.join(nopunc)
    clean_mess = [word for word in nopunc.split() if word not in stop_words]
    return clean_mess

In [81]:
data = pd.read_csv('../../data/train.csv')
data = data.dropna(subset=['Context','Response'])
data["Clean_Context"] = data["Context"].apply(text_process).apply(lambda x: ' '.join(x))
data["Clean_Response"] = data["Response"].apply(text_process).apply(lambda x: ' '.join(x))
data_unique = data.drop_duplicates(subset=['Clean_Context'])
data_unique = data_unique.drop_duplicates(subset=['Clean_Response'])
data_unique = data_unique.reset_index(drop=True)
test_data = data_unique.sample(n=10,random_state=42)
train_data = data_unique.drop(test_data.index)
train_data.to_csv('../../data/train_unique_e2p2.csv',index=False)
test_data.to_csv('../../data/test_unique_e2p2.csv',index=False)

In [80]:
if test_data['Context'].isin(train_data['Context']).any():
    print("Attention : fuite détectée dans les Contexts")
else:
    print("Pas de fuite détectée dans les Contexts")

if test_data['Response'].isin(train_data['Response']).any():
    print("Attention : fuite détectée dans les Responses")
else:
    print("Pas de fuite détectée dans les Responses")

Pas de fuite détectée dans les Contexts
Pas de fuite détectée dans les Responses


In [18]:
train_data = pd.read_csv('../../data/train_unique_e2p2.csv')
test_data = pd.read_csv('../../data/test_unique_e2p2.csv')

In [9]:
train_data["Context_clean"] = train_data["Context"].apply(text_process)
train_data["Response_clean"] = train_data["Response"].apply(text_process)
# Structure du dictionnaire : mot -> liste des discussions contenant ce type de mot
index_inverse = {}
# Nombre minimum d'occurrences pour garder une discussion
SEUIL = 1
for idx, row in train_data.iterrows():
    question = row["Context"]
    response = row["Response"]
    #combine les mots de la question et de la réponse
    mots = row["Context_clean"] + row["Response_clean"]
    # Compter le nombre d'apparitions de chaque mot
    from collections import Counter
    compteur = Counter(mots)
    for mot,count in compteur.items():
        if count >= SEUIL:
        #Si le mot n'existe pas encore dans le dictionnaire, on l'initialise
            if mot not in index_inverse:
                index_inverse[mot] = []
            #On ajoute la discussion associée à ce mot dans la liste
            index_inverse[mot].append({
                "id": idx,
                "question": question,
                "response": response
            })

In [10]:
anxiete = index_inverse.get("anxiety", [])
print("Nombre de discussion contenant le mot anxiety plus de",SEUIL,"fois :", len(anxiete))
print("\nDiscussion pour le mot 'anxiety' :\n")
#Pour un affichage plus propre
for i, item in enumerate(anxiete, 1):
    print(f"--- Exemple {i} ---")
    print(f"ID       : {item['id']}")
    print(f"Question : {item['question']}")
    print(f"Réponse  : {item['response']}\n")

Nombre de discussion contenant le mot anxiety plus de 1 fois : 120

Discussion pour le mot 'anxiety' :

--- Exemple 1 ---
ID       : 1
Question : I have so many issues to address. I have a history of sexual abuse, I’m a breast cancer survivor and I am a lifetime insomniac.    I have a long history of depression and I’m beginning to have anxiety. I have low self esteem but I’ve been happily married for almost 35 years.
   I’ve never had counseling about any of this. Do I have too many issues to address in counseling?
Réponse  : Let me start by saying there are never too many concerns that you can bring into counselling. In fact, most people who come to see me for counselling have more than one issue they would like to work on in psychotherapy and most times these are all interconnected. In counselling, we work together, collaboratively, to figure out which issues you would like to address first and then together we develop an individualized plan of care. Basically, it’s like a road map 

In [67]:
# Méthode 1 : TF-IDF
vectorizer = TfidfVectorizer(analyzer=text_process)
X_train = vectorizer.fit_transform(train_data['Context'])
def questionQuestion(question):
    X_question = vectorizer.transform([question])
    similarities = cosine_similarity(X_question, X_train).flatten()
    top_indices = similarities.argmax()
    response = train_data.iloc[top_indices]["Response"]
    score = similarities[top_indices]
    return response,score

In [68]:
# Test méthode 1 : TF-IDF
questions = test_data['Context'].tolist()
responses = test_data['Response'].tolist()
for question,response in zip(questions,responses):
    print(f"Question: {question}")
    print(f"Actual Response: {response}")
    print("Top response:")
    predicted_response,score = questionQuestion(question)
    print(f"- (Cosine Similarity: {score:.4f}) {predicted_response}")
    print("\n")

Question: I have secrets in my mind, and I don't know what to do with them. I don't want to tell my wife and mom because I don't want to hurt them. But I'm not sure how long that I can keep the secret to myself. What should I do? It's becoming annoying and making me anxious. Help me out
Actual Response: It sounds like keeping the secrets has become a problem for you now. There are several things to consider before you make a decision.- You mentioned that you don't want your wife and mom to know because you don't want to hurt them – why would it hurt them? - Is it necessary for them to know this information?- What are the consequences of either telling them the truth or not telling them? (for you and for your wife and mom).- Once you have considered these, think of what you would tell your friend if they were in your exact situation?- Also, if your wife or mom were in your situation right now, what do you think they would do themselves?- If your wife and mom were in this situation, how 

In [69]:
# Méthode 1 : Word2Vec
phrases_train = train_data["Context"].apply(text_process).tolist()
model_w2v = Word2Vec(sentences=phrases_train, vector_size=100, window=5, min_count=1, workers=4)
def vectoriser_phrase(phrase):
    mots = text_process(phrase)
    vecteurs = [model_w2v.wv[mot] for mot in mots if mot in model_w2v.wv]
    if vecteurs:
        return np.mean(vecteurs, axis=0)
    else:
        return np.zeros(model_w2v.vector_size)
X_train_w2v = np.array([vectoriser_phrase(phrase) for phrase in train_data["Context"]])
def questionQuestion_w2v(question):
    vecteur_question = vectoriser_phrase(question).reshape(1, -1)
    similarities = cosine_similarity(vecteur_question, X_train_w2v).flatten()
    top_indices = similarities.argmax()
    response = train_data.iloc[top_indices]["Response"]
    score = similarities[top_indices]
    return response,score


In [70]:
# Test méthode 1 : Word2Vec
questions = test_data['Context'].tolist()
responses = test_data['Response'].tolist()
for question,response in zip(questions,responses):
    print(f"Question: {question}")
    print(f"Actual Response: {response}")
    print("Top response:")
    predicted_response,score = questionQuestion_w2v(question)
    print(f"- (Cosine Similarity: {score:.4f}) {predicted_response}")
    print("\n")

Question: I have secrets in my mind, and I don't know what to do with them. I don't want to tell my wife and mom because I don't want to hurt them. But I'm not sure how long that I can keep the secret to myself. What should I do? It's becoming annoying and making me anxious. Help me out
Actual Response: It sounds like keeping the secrets has become a problem for you now. There are several things to consider before you make a decision.- You mentioned that you don't want your wife and mom to know because you don't want to hurt them – why would it hurt them? - Is it necessary for them to know this information?- What are the consequences of either telling them the truth or not telling them? (for you and for your wife and mom).- Once you have considered these, think of what you would tell your friend if they were in your exact situation?- Also, if your wife or mom were in your situation right now, what do you think they would do themselves?- If your wife and mom were in this situation, how 

In [71]:
# Méthode 1 : Bert
model = SentenceTransformer('all-MiniLM-L6-v2')
question_bert = model.encode(train_data["Context"].tolist())
def questionQuestion_bert(question):
    question_embedding = model.encode([question])
    cosine_similarities = cosine_similarity(question_embedding,question_bert).flatten()
    top_k_indices = cosine_similarities.argmax()
    predicted_response = train_data.iloc[top_k_indices]["Response"]
    score = cosine_similarities[top_k_indices]
    return predicted_response,score

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [72]:
# Test méthode 1 : Bert
questions = test_data['Context'].tolist()
responses = test_data['Response'].tolist()
for question,response in zip(questions,responses):
    print(f"Question: {question}")
    print(f"Actual Response: {response}")
    print("Top response:")
    predicted_response,score = questionQuestion_w2v(question)
    print(f"- (Cosine Similarity: {score:.4f}) {predicted_response}")
    print("\n")

Question: I have secrets in my mind, and I don't know what to do with them. I don't want to tell my wife and mom because I don't want to hurt them. But I'm not sure how long that I can keep the secret to myself. What should I do? It's becoming annoying and making me anxious. Help me out
Actual Response: It sounds like keeping the secrets has become a problem for you now. There are several things to consider before you make a decision.- You mentioned that you don't want your wife and mom to know because you don't want to hurt them – why would it hurt them? - Is it necessary for them to know this information?- What are the consequences of either telling them the truth or not telling them? (for you and for your wife and mom).- Once you have considered these, think of what you would tell your friend if they were in your exact situation?- Also, if your wife or mom were in your situation right now, what do you think they would do themselves?- If your wife and mom were in this situation, how 

In [73]:
# Méthde 2 : TF-IDF 
vectorizer = TfidfVectorizer(analyzer=text_process)
X_train = vectorizer.fit_transform(train_data['Response'])
def questionQuestion2(question):
    question_vector = vectorizer.transform([question])
    similarities = cosine_similarity(question_vector, X_train).flatten()
    top_indices = similarities.argmax()
    response = train_data.iloc[top_indices]["Response"]
    score = similarities[top_indices]
    return response,score

In [74]:
# Test méthode 2 : TF-IDF
questions = test_data['Context'].tolist()
responses = test_data['Response'].tolist()
for question,response in zip(questions,responses):
    print(f"Question: {question}")
    print(f"Actual Response: {response}")
    print("Top response:")
    predicted_response,score = questionQuestion2(question)
    print(f"- (Cosine Similarity: {score:.4f}) {predicted_response}")
    print("\n")

Question: I have secrets in my mind, and I don't know what to do with them. I don't want to tell my wife and mom because I don't want to hurt them. But I'm not sure how long that I can keep the secret to myself. What should I do? It's becoming annoying and making me anxious. Help me out
Actual Response: It sounds like keeping the secrets has become a problem for you now. There are several things to consider before you make a decision.- You mentioned that you don't want your wife and mom to know because you don't want to hurt them – why would it hurt them? - Is it necessary for them to know this information?- What are the consequences of either telling them the truth or not telling them? (for you and for your wife and mom).- Once you have considered these, think of what you would tell your friend if they were in your exact situation?- Also, if your wife or mom were in your situation right now, what do you think they would do themselves?- If your wife and mom were in this situation, how 

In [75]:
# Méthode 2 : Word2Vec
phrases_train = train_data["Response"].apply(text_process).tolist()
model_w2v = Word2Vec(sentences=phrases_train, vector_size=100, window=5, min_count=1, workers=4)
def vectoriser_phrase2(phrase):
    mots = text_process(phrase)
    vecteurs = [model_w2v.wv[mot] for mot in mots if mot in model_w2v.wv]
    if vecteurs:
        return np.mean(vecteurs, axis=0)
    else:
        return np.zeros(model_w2v.vector_size)
X_train_w2v = np.array([vectoriser_phrase2(phrase) for phrase in train_data["Response"]])
def questionQuestion2_w2v(question):
    vecteur_question = vectoriser_phrase2(question).reshape(1, -1)
    similarities = cosine_similarity(vecteur_question, X_train_w2v).flatten()
    top_indices = similarities.argmax()
    response = train_data.iloc[top_indices]["Response"]
    score = similarities[top_indices]
    return response,score


In [76]:
# Test méthode 2 : Word2Vec
questions = test_data['Context'].tolist()
responses = test_data['Response'].tolist()
for question,response in zip(questions,responses):
    print(f"Question: {question}")
    print(f"Actual Response: {response}")
    print("Top response:")
    predicted_response,score = questionQuestion2_w2v(question)
    print(f"- (Cosine Similarity: {score:.4f}) {predicted_response}")
    print("\n")

Question: I have secrets in my mind, and I don't know what to do with them. I don't want to tell my wife and mom because I don't want to hurt them. But I'm not sure how long that I can keep the secret to myself. What should I do? It's becoming annoying and making me anxious. Help me out
Actual Response: It sounds like keeping the secrets has become a problem for you now. There are several things to consider before you make a decision.- You mentioned that you don't want your wife and mom to know because you don't want to hurt them – why would it hurt them? - Is it necessary for them to know this information?- What are the consequences of either telling them the truth or not telling them? (for you and for your wife and mom).- Once you have considered these, think of what you would tell your friend if they were in your exact situation?- Also, if your wife or mom were in your situation right now, what do you think they would do themselves?- If your wife and mom were in this situation, how 

In [77]:
# Méthode 2 : Bert
model = SentenceTransformer('all-MiniLM-L6-v2')
question_bert = model.encode(train_data["Response"].tolist())
def questionQuestion2_bert(question):
    question_embedding = model.encode([question])
    cosine_similarities = cosine_similarity(question_embedding,question_bert).flatten()
    top_k_indices = cosine_similarities.argmax()
    predicted_response = train_data.iloc[top_k_indices]["Response"]
    score = cosine_similarities[top_k_indices]
    return predicted_response,score

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [78]:
# Test méthode 2 : Bert
questions = test_data['Context'].tolist()
responses = test_data['Response'].tolist()
for question,response in zip(questions,responses):
    print(f"Question: {question}")
    print(f"Actual Response: {response}")
    print("Top response:")
    predicted_response,score = questionQuestion2_bert(question)
    print(f"- (Cosine Similarity: {score:.4f}) {predicted_response}")
    print("\n")

Question: I have secrets in my mind, and I don't know what to do with them. I don't want to tell my wife and mom because I don't want to hurt them. But I'm not sure how long that I can keep the secret to myself. What should I do? It's becoming annoying and making me anxious. Help me out
Actual Response: It sounds like keeping the secrets has become a problem for you now. There are several things to consider before you make a decision.- You mentioned that you don't want your wife and mom to know because you don't want to hurt them – why would it hurt them? - Is it necessary for them to know this information?- What are the consequences of either telling them the truth or not telling them? (for you and for your wife and mom).- Once you have considered these, think of what you would tell your friend if they were in your exact situation?- Also, if your wife or mom were in your situation right now, what do you think they would do themselves?- If your wife and mom were in this situation, how 